# Road Extraction on DeepGlobe — Measured Evaluation & Paper Figures

Every number and figure this notebook produces is **measured** on the DeepGlobe
validation split with the trained checkpoints — nothing is simulated or hand-entered.

**What it does**
1. Rebuilds the dataset exactly the way every training run did (same archive, same
   extraction, same 90/10 split with seed 42).
2. **Verifies the split**: re-runs the training-time validation protocol and checks
   the result against the metrics stored inside each checkpoint. A match to ~3
   decimals means these are the same held-out tiles the checkpoints never trained on.
3. Evaluates every checkpoint at **native resolution** (the scale the model trains
   and deploys at): IoU, precision, recall, F1, relaxed F1 (ρ = 3 px), clDice,
   fragmentation, canopy-occluded recall; plus post-processing / TTA ablations,
   bootstrap 95% CIs and a paired Wilcoxon test.
4. Writes 300-DPI PNG + vector PDF figures, `results.json`, `per_tile_metrics.csv`
   and a LaTeX table into `/kaggle/working/paper_results/`, zipped for download.

**Before running**
* *Settings → Accelerator*: GPU (T4 or P100). *Internet*: on.
* *Add Input* → your checkpoints dataset. Any of these filenames are recognized:
  `best_model_v2.pth` (final model), `best_model_v2_epoch18_iou0159.pth`,
  `best_model_v2_collapsed_epoch46.pth`, `best_model_new.pth` (June baseline).
* *(Optional)* *Add Input* → *Notebooks* → your training notebook, so its
  `wandb_local.json` log is picked up for the training-curve figure.
* *Run All* (≈45–75 min, mostly CPU post-processing), then download
  `paper_results.zip` from the link printed by the last cell.

In [ ]:
# ── 1. Setup: dependencies and the dataset (identical source to every training run) ──
import os, sys, json, time, math, random, hashlib, subprocess, glob, zipfile, types, warnings
warnings.filterwarnings("ignore")

IN_KAGGLE = os.path.isdir("/kaggle/input")
WORK = os.environ.get("EVAL_WORK", "/kaggle/working")
DATA_DIR = os.environ.get("EVAL_DATA", os.path.join(WORK, "dataset"))
INPUT_ROOTS = os.environ.get("EVAL_INPUT", "/kaggle/input").split(os.pathsep)
NUM_WORKERS = 2 if IN_KAGGLE else 0   # DataLoader workers (0 for Windows smoke tests: no fork)
OUT_DIR = os.environ.get("EVAL_OUT", os.path.join(WORK, "paper_results"))
MAX_TILES = int(os.environ.get("EVAL_MAX_TILES", "0"))   # 0 = the full split. >0 is for smoke tests only.
FILE_ID = "1OI0XJ1-ejxd0JS45hBzJbAYe9_2hJJwN"               # the archive every training run downloaded
SEED = 42
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

if IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "albumentations", "scikit-image"], check=True)

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    import gdown
    zip_path = os.path.join(WORK, "archive.zip")
    if not os.path.exists(zip_path):
        gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", zip_path, quiet=False)
    # Same extraction command as the training notebooks: the directory listing order it
    # produces is what the seed-42 shuffle was applied to.
    subprocess.run(["unzip", "-q", zip_path, "-d", DATA_DIR], check=True)
TRAIN_DIR = os.path.join(DATA_DIR, "train")
print("train dir:", TRAIN_DIR, "| files:", len(os.listdir(TRAIN_DIR)))

In [ ]:
# ── 2. Imports and environment record ──
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.morphology import skeletonize
from skimage.filters import threshold_otsu
from scipy import stats as sstats
import matplotlib
if not IN_KAGGLE:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False


class _OriginalCanopyShadowDropout(A.ImageOnlyTransform):
    """The exact constructor call the training runs used, to check whether the canopy
    augmentation was actually active in this albumentations version."""
    def __init__(self, always_apply=False, p=0.5):
        super().__init__(always_apply, p)


try:
    _orig_p = float(_OriginalCanopyShadowDropout(p=0.5).p)
except Exception as e:  # albumentations versions that reject the extra positional arg
    _orig_p = f"constructor raised {type(e).__name__}: {e}"

ENV = {
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch": torch.__version__, "albumentations": A.__version__, "opencv": cv2.__version__,
    "canopy_aug_effective_p_with_training_code": _orig_p,
}
print(json.dumps(ENV, indent=2))
if isinstance(_orig_p, float) and _orig_p == 0.0:
    print("NOTE: with this albumentations version the training notebooks' CanopyShadowDropout "
          "had p = 0.0, i.e. canopy augmentation was NOT applied during those runs.")

In [ ]:
# ── 3. Model, loss and post-processing code (inlined verbatim from the repository) ──
# ---- inlined from backend/src/models/mobilevit_v2.py ----
"""
mobilevit_v2.py — Novel Ultra-Lightweight Encoder-Decoder for Rural Road Extraction

Architecture:  MobileViT v2 backbone with two domain-specific novelties
┌────────────────────────────────────────────────────────────────────────────┐
│ Novelty 1 — Strip Convolutions (1×3 → 3×1)                              │
│   Factorized directional filters act as 'road-shaped scanners',          │
│   injecting inductive bias for elongated tubular structures while        │
│   reducing parameters by ~33 % vs standard 3×3 convolutions.             │
│                                                                          │
│ Novelty 2 — Channel Shift (Zero-Parameter Receptive Field Expansion)     │
│   25 % of feature channels are physically displaced by 2 pixels in       │
│   each cardinal direction before transformer blocks, widening the        │
│   effective receptive field at zero computational / parameter cost.       │
└────────────────────────────────────────────────────────────────────────────┘

Input:   (B, 3, 256, 256) RGB satellite tiles
Output:  (B, 1, 256, 256) binary road probability mask (Sigmoid-activated)
Target:  < 3 M trainable parameters (~1.6 M at width_mult=1.0)

Reference:
    Mehta & Rastegari, "Separable Self-attention for Mobile Vision
    Transformers" (MobileViT v2), Apple ML Research, 2022.

Author:  Member 1 — Lead Architect
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ───────────────────────────────────────────────────────────────────────────
#  Utility
# ───────────────────────────────────────────────────────────────────────────

def _make_divisible(v: float, divisor: int = 8, min_value: int = None) -> int:
    """Round channel count to nearest *divisor* for hardware efficiency."""
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    # Avoid rounding down by more than 10 %
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


# ═══════════════════════════════════════════════════════════════════════════
#  NOVELTY 1 — Strip Convolution
# ═══════════════════════════════════════════════════════════════════════════

class StripConv(nn.Module):
    """
    Factorized 3×3 convolution decomposed into sequential 1-D directional
    filters: horizontal (1×3) followed by vertical (3×1).

    Motivation
    ----------
    Roads in satellite imagery are predominantly linear / tubular structures.
    Standard 3×3 kernels waste capacity on isotropic features.  Strip
    convolutions inject a structural prior for elongated shapes by
    independently scanning horizontal and vertical orientations — acting as
    a **road-shaped scanner** that filters out background jungle noise.

    Parameter savings
    -----------------
    Standard 3×3:   C_in × C_out × 9
    Strip (1×3→3×1): C_in × C_out × 3  +  C_out × C_out × 3
    When C_in ≈ C_out → ~33 % reduction.

    Parameters
    ----------
    in_channels  : Number of input channels.
    out_channels : Number of output channels.
    stride       : Spatial downsampling factor (applied across both dims).
    """

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        # Horizontal scan — captures road-like patterns along the x-axis
        self.conv_h = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=(1, 3), stride=(1, stride), padding=(0, 1), bias=False,
        )
        self.bn_h = nn.BatchNorm2d(out_channels)

        # Vertical scan — captures road-like patterns along the y-axis
        self.conv_v = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=(3, 1), stride=(stride, 1), padding=(1, 0), bias=False,
        )
        self.bn_v = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act(self.bn_h(self.conv_h(x)))
        x = self.act(self.bn_v(self.conv_v(x)))
        return x


# ═══════════════════════════════════════════════════════════════════════════
#  NOVELTY 2 — Channel Shift
# ═══════════════════════════════════════════════════════════════════════════

class ChannelShift(nn.Module):
    """
    Zero-parameter spatial displacement of feature channels.

    Before transformer blocks a fraction of channels are physically shifted
    by ``shift_pixels`` in each of the four cardinal directions (↑ ↓ ← →).
    This lets each spatial position access information from neighbouring
    pixels **without** any learned parameters or FLOPs (beyond the memory
    copy).

    Effect
    ------
    Effectively widens the receptive field by 2 × shift_pixels in each
    direction, giving the downstream linear attention richer local context
    to aggregate over.  At 4× downsampling and shift_pixels = 2, the model
    gains an extra ±8 pixel reach in the original 256×256 image space —
    roughly the width of a rural road.

    Parameters
    ----------
    shift_pixels   : Number of pixels to shift (default 2).
    shift_fraction : Fraction of channels to shift (default 0.25).
                     Split equally across four directions; the remaining
                     channels pass through unchanged.

    Note
    ----
    This module has **ZERO** trainable parameters.
    """

    def __init__(self, shift_pixels: int = 2, shift_fraction: float = 0.25):
        super().__init__()
        self.shift_pixels = shift_pixels
        self.shift_fraction = shift_fraction

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        s = self.shift_pixels
        n_shifted = int(C * self.shift_fraction)
        per_dir = n_shifted // 4

        if per_dir == 0 or s == 0:
            return x

        # Split into directional groups + identity pass-through
        c_up    = x[:, 0 * per_dir : 1 * per_dir]
        c_down  = x[:, 1 * per_dir : 2 * per_dir]
        c_left  = x[:, 2 * per_dir : 3 * per_dir]
        c_right = x[:, 3 * per_dir : 4 * per_dir]
        c_id    = x[:, 4 * per_dir :]                 # identity (unchanged)

        # Shift-and-pad: displace spatial content, zero-fill vacated edges
        c_up    = F.pad(c_up[:, :, s:, :],     (0, 0, 0, s))   # ↑ pad bottom
        c_down  = F.pad(c_down[:, :, :-s, :],  (0, 0, s, 0))   # ↓ pad top
        c_left  = F.pad(c_left[:, :, :, s:],   (0, s, 0, 0))   # ← pad right
        c_right = F.pad(c_right[:, :, :, :-s], (s, 0, 0, 0))   # → pad left

        return torch.cat([c_up, c_down, c_left, c_right, c_id], dim=1)

    def extra_repr(self) -> str:
        return (f"shift_pixels={self.shift_pixels}, "
                f"shift_fraction={self.shift_fraction}, "
                f"parameters=0")


# ═══════════════════════════════════════════════════════════════════════════
#  MobileNet V2 Inverted Residual Block
# ═══════════════════════════════════════════════════════════════════════════

class InvertedResidual(nn.Module):
    """
    MobileNetV2 inverted residual bottleneck (Sandler et al., 2018).

    Pipeline:  1×1 expand → 3×3 depthwise → 1×1 linear project.
    Residual connection when stride = 1 and in_channels == out_channels.
    """

    def __init__(self, in_channels: int, out_channels: int,
                 stride: int = 1, expand_ratio: int = 2):
        super().__init__()
        mid = in_channels * expand_ratio
        self.use_residual = (stride == 1 and in_channels == out_channels)

        layers = []
        # Pointwise expansion (skip when expand_ratio == 1)
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, mid, 1, bias=False),
                nn.BatchNorm2d(mid),
                nn.SiLU(inplace=True),
            ])
        # Depthwise separable convolution
        layers.extend([
            nn.Conv2d(mid, mid, 3, stride=stride, padding=1,
                      groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(inplace=True),
            # Linear projection (no activation — linear bottleneck)
            nn.Conv2d(mid, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        ])
        self.conv = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_residual:
            return x + self.conv(x)
        return self.conv(x)


# ═══════════════════════════════════════════════════════════════════════════
#  Linear Self-Attention  (MobileViT v2 Separable Attention)
# ═══════════════════════════════════════════════════════════════════════════

class LinearSelfAttention(nn.Module):
    """
    Separable Self-Attention from MobileViT v2 (Mehta, 2022).

    Replaces O(N²) scaled dot-product attention with O(N·d) linear
    attention through a global context vector:

        1. Project input → Q (d),  K (1),  V (d)
        2. Context scores:   α = softmax(K, dim=tokens)       — (B, N, 1)
        3. Global context:   c = Σ_n(α_n · V_n)               — (B, 1, d)
        4. Output:           O = ReLU(Q) ⊙ broadcast(c)       — (B, N, d)

    The **scalar** K projection enables global context aggregation at
    minimal cost while Q gating ensures token-specific output modulation.

    Parameters
    ----------
    embed_dim    : Transformer embedding dimension.
    attn_dropout : Dropout probability on context scores.
    """

    def __init__(self, embed_dim: int, attn_dropout: float = 0.0):
        super().__init__()
        self.embed_dim = embed_dim
        # Project to  Q(d) + K(1) + V(d) = 2d + 1
        self.qkv = nn.Linear(embed_dim, 2 * embed_dim + 1, bias=True)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=True)
        self.attn_drop = nn.Dropout(attn_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, N, d)
        qkv = self.qkv(x)
        q, k, v = qkv.split([self.embed_dim, 1, self.embed_dim], dim=-1)

        # Aggregate global context via weighted sum of values
        context_scores = F.softmax(k, dim=1)                          # (B, N, 1)
        context_scores = self.attn_drop(context_scores)
        context_vector = (context_scores * v).sum(dim=1, keepdim=True)  # (B, 1, d)

        # Modulate queries with the global context
        out = F.relu(q) * context_vector                               # (B, N, d)
        return self.out_proj(out)


# ═══════════════════════════════════════════════════════════════════════════
#  Transformer Block
# ═══════════════════════════════════════════════════════════════════════════

class TransformerBlock(nn.Module):
    """Pre-LayerNorm Transformer block with Linear Self-Attention and FFN."""

    def __init__(self, embed_dim: int, ffn_ratio: float = 2.0,
                 dropout: float = 0.0, attn_dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LinearSelfAttention(embed_dim, attn_dropout)
        self.norm2 = nn.LayerNorm(embed_dim)

        ffn_dim = int(embed_dim * ffn_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


# ═══════════════════════════════════════════════════════════════════════════
#  MobileViT v2 Block  (with Channel Shift + Strip-Aware Processing)
# ═══════════════════════════════════════════════════════════════════════════

class MobileViTv2Block(nn.Module):
    """
    MobileViT v2 block with integrated Channel Shift (Novelty 2).

    Pipeline
    --------
    ① Channel Shift — zero-cost receptive field expansion
    ② 3×3 depthwise conv → 1×1 pointwise (local representation)
    ③ Unfold into patches → L × Transformer layers → fold back (global)
    ④ 1×1 projection back to input channel count
    ⑤ Concatenate with original input → 1×1 fusion conv

    The block preserves spatial dimensions (H, W) and channel count.

    Parameters
    ----------
    in_channels            : Number of input / output channels.
    transformer_dim        : Internal transformer embedding dimension.
    n_transformer_layers   : Number of stacked transformer blocks.
    patch_size             : Spatial patch size for unfold / fold (default 2).
    ffn_ratio              : FFN expansion ratio inside transformer.
    dropout                : FFN dropout probability.
    attn_dropout           : Attention weight dropout probability.
    shift_pixels           : Channel Shift displacement (Novelty 2).
    shift_fraction         : Fraction of channels to shift (Novelty 2).
    """

    def __init__(
        self,
        in_channels: int,
        transformer_dim: int,
        n_transformer_layers: int = 2,
        patch_size: int = 2,
        ffn_ratio: float = 2.0,
        dropout: float = 0.0,
        attn_dropout: float = 0.0,
        shift_pixels: int = 2,
        shift_fraction: float = 0.25,
    ):
        super().__init__()
        self.patch_h = patch_size
        self.patch_w = patch_size

        # ① Channel Shift (Novelty 2)
        self.channel_shift = ChannelShift(shift_pixels, shift_fraction)

        # ② Local representation
        self.local_rep = nn.Sequential(
            # Depthwise conv — local spatial features
            nn.Conv2d(in_channels, in_channels, 3, padding=1,
                      groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=True),
            # Pointwise projection to transformer dimension
            nn.Conv2d(in_channels, transformer_dim, 1, bias=False),
            nn.BatchNorm2d(transformer_dim),
        )

        # ③ Global representation (transformer stack)
        self.transformers = nn.Sequential(*[
            TransformerBlock(transformer_dim, ffn_ratio, dropout, attn_dropout)
            for _ in range(n_transformer_layers)
        ])
        self.post_norm = nn.LayerNorm(transformer_dim)

        # ④ Project back to input channel count
        self.proj = nn.Sequential(
            nn.Conv2d(transformer_dim, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
        )

        # ⑤ Fuse original input + transformer-processed output
        self.fusion = nn.Sequential(
            nn.Conv2d(2 * in_channels, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.SiLU(inplace=True),
        )

    # ── Patch unfold / fold ────────────────────────────────────────────

    def _unfold(self, x: torch.Tensor):
        """
        Unfold spatial dims into patch tokens for transformer processing.

        For each pixel position (i, j) within a patch, all patches across
        the image are gathered as a token sequence.  This allows the
        transformer to model *global* relationships for every local
        sub-pixel position.

        (B, C, H, W)  →  (B · ph · pw,  N_patches,  C)
        where N_patches = (H / ph) × (W / pw).
        """
        B, C, H, W = x.shape
        ph, pw = self.patch_h, self.patch_w
        n_h, n_w = H // ph, W // pw

        x = x.reshape(B, C, n_h, ph, n_w, pw)
        x = x.permute(0, 3, 5, 1, 2, 4)          # (B, ph, pw, C, n_h, n_w)
        x = x.reshape(B * ph * pw, C, n_h * n_w)  # (B·ph·pw, C, N)
        x = x.permute(0, 2, 1)                    # (B·ph·pw, N, C)
        return x, (B, n_h, n_w)

    def _fold(self, x: torch.Tensor, info: tuple, C: int):
        """
        Fold patch tokens back into a spatial feature map.

        (B · ph · pw,  N_patches,  C)  →  (B, C, H, W)
        """
        B, n_h, n_w = info
        ph, pw = self.patch_h, self.patch_w

        x = x.permute(0, 2, 1)                            # (B·ph·pw, C, N)
        x = x.reshape(B, ph, pw, C, n_h, n_w)
        x = x.permute(0, 3, 4, 1, 5, 2)                   # (B, C, n_h, ph, n_w, pw)
        x = x.reshape(B, C, n_h * ph, n_w * pw)
        return x

    # ── Forward ────────────────────────────────────────────────────────

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C_in, H, W = x.shape
        ph, pw = self.patch_h, self.patch_w

        # Pad spatial dims to be divisible by patch_size (safety)
        pad_h = (ph - H % ph) % ph
        pad_w = (pw - W % pw) % pw
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, pad_w, 0, pad_h))

        identity = x                              # save for fusion ⑤

        # ① Channel Shift
        x_shifted = self.channel_shift(x)

        # ② Local representation
        local_out = self.local_rep(x_shifted)
        C_t = local_out.shape[1]

        # ③ Unfold → Transformer → Fold
        tokens, fold_info = self._unfold(local_out)
        tokens = self.transformers(tokens)
        tokens = self.post_norm(tokens)
        global_out = self._fold(tokens, fold_info, C_t)

        # ④ Project back to in_channels
        global_out = self.proj(global_out)

        # ⑤ Fuse original + global
        fused = self.fusion(torch.cat([identity, global_out], dim=1))

        # Remove padding if it was applied
        if pad_h > 0 or pad_w > 0:
            fused = fused[:, :, :H, :W]

        return fused


# ═══════════════════════════════════════════════════════════════════════════
#  Attention Gate (Skip Connection Attention Gating)
# ═══════════════════════════════════════════════════════════════════════════

class AttentionGate(nn.Module):
    """
    Additive Attention Gate for skip connections (Oktay et al., Attention U-Net).
    Filters background clutter (rooftops, field boundaries) from skip features
    before concatenating with upsampled decoder representations.
    """
    def __init__(self, gate_channels: int, skip_channels: int, inter_channels: int = None):
        super().__init__()
        inter_channels = inter_channels or max(skip_channels // 2, 8)
        self.W_gate = nn.Sequential(
            nn.Conv2d(gate_channels, inter_channels, 1, bias=True),
            nn.BatchNorm2d(inter_channels),
        )
        self.W_skip = nn.Sequential(
            nn.Conv2d(skip_channels, inter_channels, 1, bias=True),
            nn.BatchNorm2d(inter_channels),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_channels, 1, 1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.act = nn.SiLU(inplace=True)

    def forward(self, gate: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        g = self.W_gate(gate)
        s = self.W_skip(skip)
        attn = self.psi(self.act(g + s))     # spatial attention mask in [0, 1]
        return skip * attn


# ═══════════════════════════════════════════════════════════════════════════
#  Full Model — MobileViT v2 Encoder-Decoder for Road Segmentation
# ═══════════════════════════════════════════════════════════════════════════

class MobileViT_v2(nn.Module):
    """
    Ultra-lightweight encoder-decoder for binary road segmentation with Attention Gates.
    """

    def __init__(self, num_classes: int = 1, width_mult: float = 1.0,
                 attention_gates: bool = True):
        super().__init__()
        # attention_gates=False reproduces the original (June) architecture that
        # best_model.pth / best_model_new.pth were trained with -- plain U-Net-style
        # skip concatenation. Loading those checkpoints into the gated model with
        # strict=False silently leaves the gates randomly initialized.
        self.attention_gates = attention_gates

        def _c(channels: int) -> int:
            """Scale channel count by width multiplier, round to nearest 8."""
            return _make_divisible(channels * width_mult)

        # ─── ENCODER ────────────────────────────────────────────────
        # Stem: Strip Convolution (Novelty 1 — road-shaped scanner)
        self.stem = StripConv(3, _c(32), stride=2)         # 256 → 128

        # Stage 1: MV2 downsampling
        self.enc1 = InvertedResidual(_c(32), _c(64), stride=2)  # 128 → 64

        # Stage 2: MobileViT v2 (Channel Shift) + downsample
        self.enc2_mvit = MobileViTv2Block(
            _c(64), transformer_dim=_c(96), n_transformer_layers=2,
        )
        self.enc2_down = InvertedResidual(_c(64), _c(96), stride=2)  # 64 → 32

        # Stage 3: MobileViT v2 (Channel Shift) + downsample
        self.enc3_mvit = MobileViTv2Block(
            _c(96), transformer_dim=_c(144), n_transformer_layers=2,
        )
        self.enc3_down = InvertedResidual(_c(96), _c(128), stride=2)  # 32 → 16

        # Bottleneck: deepest MobileViT v2 block (3 transformer layers)
        self.bottleneck = MobileViTv2Block(
            _c(128), transformer_dim=_c(192), n_transformer_layers=3,
        )

        # ─── ATTENTION GATES FOR SKIPS ──────────────────────────────
        if attention_gates:
            self.gate3 = AttentionGate(gate_channels=_c(128), skip_channels=_c(96))
            self.gate2 = AttentionGate(gate_channels=_c(96), skip_channels=_c(64))
            self.gate1 = AttentionGate(gate_channels=_c(64), skip_channels=_c(32))

        # ─── DECODER ────────────────────────────────────────────────
        # Each stage: bilinear upsample → concat gated skip → StripConv (Novelty 1)
        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear",
                               align_corners=False)
        self.dec3 = StripConv(_c(128) + _c(96), _c(96))    # 16 → 32

        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear",
                               align_corners=False)
        self.dec2 = StripConv(_c(96) + _c(64), _c(64))     # 32 → 64

        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear",
                               align_corners=False)
        self.dec1 = StripConv(_c(64) + _c(32), _c(32))     # 64 → 128

        self.up0 = nn.Upsample(scale_factor=2, mode="bilinear",
                               align_corners=False)                     # 128 → 256

        # ─── SEGMENTATION HEAD ──────────────────────────────────────
        self.head = nn.Conv2d(_c(32), num_classes, kernel_size=1, bias=True)

        # ─── Weight Initialisation ──────────────────────────────────
        self._init_weights()

    # ── Init ───────────────────────────────────────────────────────────

    def _init_weights(self):
        """Kaiming init for convs, truncated-normal for linear layers."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                        nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    # ── Forward ────────────────────────────────────────────────────────

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (B, 3, H, W) RGB input tensor.  H and W must be divisible by 16.
            Designed for H = W = 256.

        Returns
        -------
        (B, num_classes, H, W) sigmoid-activated probability map in [0, 1].
        """
        # ── Encoder ──
        s1 = self.stem(x)              # skip1: (_c(32), 128, 128)
        e1 = self.enc1(s1)             #         (_c(64),  64,  64)

        s2 = self.enc2_mvit(e1)        # skip2: (_c(64),  64,  64)
        e2 = self.enc2_down(s2)        #         (_c(96),  32,  32)

        s3 = self.enc3_mvit(e2)        # skip3: (_c(96),  32,  32)
        e3 = self.enc3_down(s3)        #         (_c(128), 16,  16)

        bn = self.bottleneck(e3)       #         (_c(128), 16,  16)

        # ── Decoder with (optionally attention-gated) skip connections ──
        up3 = self.up3(bn)
        if self.attention_gates:
            s3 = self.gate3(up3, s3)
        d3 = self.dec3(torch.cat([up3, s3], dim=1))           # 32 × 32

        up2 = self.up2(d3)
        if self.attention_gates:
            s2 = self.gate2(up2, s2)
        d2 = self.dec2(torch.cat([up2, s2], dim=1))           # 64 × 64

        up1 = self.up1(d2)
        if self.attention_gates:
            s1 = self.gate1(up1, s1)
        d1 = self.dec1(torch.cat([up1, s1], dim=1))           # 128 × 128

        out = self.head(self.up0(d1))                         # 256 × 256
        return out


    # ── Convenience ────────────────────────────────────────────────────

    @property
    def num_parameters(self) -> int:
        """Total number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def load_mobilevit_checkpoint(checkpoint_path: str, device="cpu", width_mult: float = 1.0):
    """
    Builds the architecture a checkpoint was actually trained with (gated vs.
    ungated skips, detected from its keys) and loads it strictly.

    Returns (model, checkpoint_metadata) where metadata holds whatever training
    stored alongside the weights (epoch, val_iou, ...), or {} for raw state dicts.
    """
    checkpoint = torch.load(checkpoint_path, map_location=device)
    is_wrapped = isinstance(checkpoint, dict) and "model_state_dict" in checkpoint
    state_dict = checkpoint["model_state_dict"] if is_wrapped else checkpoint
    has_gates = any(k.startswith("gate") for k in state_dict)

    model = MobileViT_v2(num_classes=1, width_mult=width_mult, attention_gates=has_gates)
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()

    metadata = {k: v for k, v in checkpoint.items()
                if k not in ("model_state_dict", "optimizer_state_dict")} if is_wrapped else {}
    return model, metadata


# ═══════════════════════════════════════════════════════════════════════════
#  Quick verification
# ═══════════════════════════════════════════════════════════════════════════
# ---- end backend/src/models/mobilevit_v2.py ----

# ---- inlined from backend/src/utils/graph_postprocess.py ----
import cv2
import numpy as np
import scipy.ndimage as ndimage
from typing import List, Tuple

try:
    from skimage.morphology import skeletonize
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False

def hysteresis_threshold(probs: np.ndarray, high_thresh: float = 0.35, low_thresh: float = 0.12) -> np.ndarray:
    """
    Hysteresis Thresholding: Retains weak road predictions (>= low_thresh) if they are connected
    to high-confidence road regions (>= high_thresh). Vectorized for ultra-fast, memory-efficient execution.
    """
    strong = probs >= high_thresh
    weak = (probs >= low_thresh) & (probs < high_thresh)
    
    total_candidate = (strong | weak).astype(np.uint8)
    num_labels, labels = cv2.connectedComponents(total_candidate, connectivity=8)
    
    if num_labels <= 1:
        return (strong * 255).astype(np.uint8)
        
    strong_labels = np.unique(labels[strong])
    strong_labels = strong_labels[strong_labels > 0]
    
    keep_mask = np.isin(labels, strong_labels)
    output_mask = np.zeros_like(total_candidate, dtype=np.uint8)
    output_mask[keep_mask] = 255
            
    return output_mask

def find_skeleton_endpoints(skel: np.ndarray) -> List[Tuple[int, int, np.ndarray]]:
    """
    Finds endpoints in a 1-pixel binary skeleton and computes their outward tangent direction vectors.
    Returns list of (y, x, direction_vector).
    """
    skel_bool = (skel > 0).astype(np.uint8)
    kernel = np.array([[1, 1, 1],
                       [1, 10, 1],
                       [1, 1, 1]], dtype=np.uint8)
    
    filtered = cv2.filter2D(skel_bool, -1, kernel)
    ey, ex = np.where(filtered == 11)
    
    endpoints = []
    h, w = skel.shape
    
    for y, x in zip(ey, ex):
        patch_y1, patch_y2 = max(0, y - 4), min(h, y + 5)
        patch_x1, patch_x2 = max(0, x - 4), min(w, x + 5)
        
        py, px = np.where(skel_bool[patch_y1:patch_y2, patch_x1:patch_x2] > 0)
        py = py + patch_y1
        px = px + patch_x1
        
        if len(py) > 1:
            dy = float(y - np.mean(py[py != y])) if any(py != y) else 0.0
            dx = float(x - np.mean(px[px != x])) if any(px != x) else 0.0
            norm = np.hypot(dx, dy)
            if norm > 1e-5:
                dir_vec = np.array([dx / norm, dy / norm])
            else:
                dir_vec = np.array([0.0, 0.0])
        else:
            dir_vec = np.array([0.0, 0.0])
            
        endpoints.append((y, x, dir_vec))
        
    return endpoints

def connect_canopy_gaps(
    binary_mask: np.ndarray,
    max_gap_dist: float = 220.0,
    max_angle_deg: float = 65.0,
    road_width: int = 6,
    border_margin: int = 16,
) -> np.ndarray:
    """
    Multi-Strategy Graph-Based Post-Processing:
    1. Extracts topological 1-pixel skeleton using scikit-image skeletonize (or cv2.ximgproc as fallback).
    2. Bridges facing dead-end endpoints across wide tree canopy gaps (up to max_gap_dist).
    3. Connects dead-end endpoints to nearby main road segments (T-junction completion).

    Endpoints within ``border_margin`` px of the tile edge are ignored: there a road simply
    leaves the tile, it is not interrupted. Treating those as dead ends made the bridging
    draw spurious straight "roads" along the tile border between neighbouring exits.
    ``border_margin=0`` reproduces the original behaviour.
    """
    mask_out = binary_mask.copy()
    h, w = mask_out.shape
    
    # 1. Extract 1-pixel topological skeleton
    if HAS_SKIMAGE:
        skel = skeletonize(mask_out > 0).astype(np.uint8)
    elif hasattr(cv2, 'ximgproc'):
        skel = cv2.ximgproc.thinning(mask_out)
    else:
        # Morphological fallback
        skel = (mask_out > 0).astype(np.uint8)
        element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
        done = False
        skel_acc = np.zeros(mask_out.shape, dtype=np.uint8)
        img_temp = skel.copy()
        while not done:
            eroded = cv2.erode(img_temp, element)
            temp = cv2.dilate(eroded, element)
            temp = cv2.subtract(img_temp, temp)
            skel_acc = cv2.bitwise_or(skel_acc, temp)
            img_temp = eroded.copy()
            if cv2.countNonZero(img_temp) == 0:
                done = True
        skel = skel_acc

    endpoints = [(y, x, v) for y, x, v in find_skeleton_endpoints(skel)
                 if border_margin <= y < h - border_margin and border_margin <= x < w - border_margin]
    n_pts = len(endpoints)
    if n_pts == 0:
        return mask_out

    connected_pairs = []
    connected_endpoints = set()
    
    cos_threshold = np.cos(np.radians(max_angle_deg))
    
    # --- Strategy A: Endpoint-to-Endpoint Collinear & Facing Pair Connection ---
    for i in range(n_pts):
        y1, x1, v1 = endpoints[i]
        best_j = None
        best_score = float('inf')
        
        for j in range(i + 1, n_pts):
            y2, x2, v2 = endpoints[j]
            dist = float(np.hypot(x2 - x1, y2 - y1))
            if dist > max_gap_dist or dist < 5.0:
                continue
                
            gap_vec = np.array([(x2 - x1) / dist, (y2 - y1) / dist])
            
            dot1 = float(np.dot(v1, gap_vec)) if np.linalg.norm(v1) > 0 else 0.8
            dot2 = float(np.dot(v2, -gap_vec)) if np.linalg.norm(v2) > 0 else 0.8
            
            if dot1 >= cos_threshold and dot2 >= cos_threshold:
                alignment_penalty = (2.0 - dot1 - dot2) * 50.0
                score = dist + alignment_penalty
                if score < best_score:
                    best_score = score
                    best_j = j
                    
        if best_j is not None:
            y2, x2, _ = endpoints[best_j]
            connected_pairs.append(((x1, y1), (x2, y2)))
            connected_endpoints.add(i)
            connected_endpoints.add(best_j)

    # --- Strategy B: Endpoint-to-Road Edge Connection (T-Junction Completion) ---
    skel_y, skel_x = np.where(skel > 0)
    if len(skel_x) > 0:
        skel_pts = np.column_stack((skel_x, skel_y))
        
        for i in range(n_pts):
            if i in connected_endpoints:
                continue
            y1, x1, v1 = endpoints[i]
            if np.linalg.norm(v1) == 0:
                continue
                
            dists_to_skel = np.hypot(skel_pts[:, 0] - x1, skel_pts[:, 1] - y1)
            valid_idx = dists_to_skel > 25.0  # Outside local 25px radius
            
            if not np.any(valid_idx):
                continue
                
            cand_pts = skel_pts[valid_idx]
            cand_dists = dists_to_skel[valid_idx]
            
            within_dist = cand_dists <= max_gap_dist
            if not np.any(within_dist):
                continue
                
            cand_pts = cand_pts[within_dist]
            cand_dists = cand_dists[within_dist]
            
            best_cand = None
            best_cand_score = float('inf')
            
            for (cx, cy), d in zip(cand_pts, cand_dists):
                g_vec = np.array([(cx - x1) / d, (cy - y1) / d])
                dot = float(np.dot(v1, g_vec))
                if dot >= cos_threshold:
                    score = d + (1.0 - dot) * 60.0
                    if score < best_cand_score:
                        best_cand_score = score
                        best_cand = (cx, cy)
                        
            if best_cand is not None:
                connected_pairs.append(((x1, y1), best_cand))

    # Draw connecting road strokes
    for pt1, pt2 in connected_pairs:
        cv2.line(mask_out, pt1, pt2, 255, thickness=road_width)
        
    return mask_out
# ---- end backend/src/utils/graph_postprocess.py ----

# ---- inlined from backend/src/utils/loss.py ----
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Tuple

# =====================================================================
#  Soft Morphological Primitives
# =====================================================================

def soft_erode(img: torch.Tensor) -> torch.Tensor:
    if img.dim() == 3:
        img = img.unsqueeze(1)
    p_v = -F.max_pool2d(-img, kernel_size=(3, 1), stride=1, padding=(1, 0))
    p_h = -F.max_pool2d(-img, kernel_size=(1, 3), stride=1, padding=(0, 1))
    return torch.min(p_v, p_h)

def soft_dilate(img: torch.Tensor) -> torch.Tensor:
    if img.dim() == 3:
        img = img.unsqueeze(1)
    return F.max_pool2d(img, kernel_size=3, stride=1, padding=1)

def soft_open(img: torch.Tensor) -> torch.Tensor:
    return soft_dilate(soft_erode(img))

def soft_skel(img: torch.Tensor, num_iter: int = 10) -> torch.Tensor:
    if img.dim() == 3:
        img = img.unsqueeze(1)
    img1 = soft_open(img)
    skel = F.relu(img - img1)
    for _ in range(num_iter):
        img = soft_erode(img)
        img1 = soft_open(img)
        delta = F.relu(img - img1)
        skel = skel + F.relu(delta - skel * delta)
    return skel

# =====================================================================
#  Soft Centerline-Dice (clDice) Loss
# =====================================================================

class SoftClDiceLoss(nn.Module):
    def __init__(self, num_iter: int = 10, smooth: float = 1.0):
        super().__init__()
        self.num_iter = num_iter
        self.smooth = smooth

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if pred.dim() == 3:
            pred = pred.unsqueeze(1)
        if target.dim() == 3:
            target = target.unsqueeze(1)

        skel_pred = soft_skel(torch.sigmoid(pred), self.num_iter)
        skel_target = soft_skel(target, self.num_iter)

        tprec_num = (skel_pred * target).sum(dim=(1, 2, 3)) + self.smooth
        tprec_den = skel_pred.sum(dim=(1, 2, 3)) + self.smooth
        tprec = tprec_num / tprec_den

        tsens_num = (skel_target * torch.sigmoid(pred)).sum(dim=(1, 2, 3)) + self.smooth
        tsens_den = skel_target.sum(dim=(1, 2, 3)) + self.smooth
        tsens = tsens_num / tsens_den

        cl_dice = 2.0 * (tprec * tsens) / (tprec + tsens + 1e-7)
        return (1.0 - cl_dice).mean()

# =====================================================================
#  Soft Dice Loss
# =====================================================================

class SoftDiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred = torch.sigmoid(pred)  # ensure probabilities
        if pred.dim() == 3:
            pred = pred.unsqueeze(1)
        if target.dim() == 3:
            target = target.unsqueeze(1)

        intersection = (pred * target).sum(dim=(1, 2, 3))
        cardinality = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return (1.0 - dice).mean()

# =====================================================================
#  RoadExtractionLoss (BCEWithLogits + clDice with pos_weight & decay)
# =====================================================================

class RoadExtractionLoss(nn.Module):
    def __init__(
        self,
        total_epochs: int,
        alpha_start: float = 0.5,
        alpha_end: float = 0.15,
        dice_weight: float = 0.35,
        num_iter: int = 10,
        smooth: float = 1.0,
        pos_weight: float = 2.0,
        decay_power: float = 0.5,
        alpha_decay_epochs: int | None = None,
    ):
        super().__init__()
        self.total_epochs = max(total_epochs, 1)
        self.decay_epochs = max(alpha_decay_epochs or total_epochs, 1)
        self.alpha_start = alpha_start
        self.alpha_end = alpha_end
        self.alpha = alpha_start
        self.dice_weight = dice_weight
        self.decay_power = decay_power

        pw_tensor = torch.tensor([pos_weight]) if isinstance(pos_weight, (int, float)) else pos_weight
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pw_tensor)
        self.dice = SoftDiceLoss(smooth=smooth)
        self.cldice = SoftClDiceLoss(num_iter=num_iter, smooth=smooth)

    def get_alpha(self) -> float:
        return self.alpha

    def update_alpha(self, epoch: int) -> float:
        # Non-linear decay decoupled from total epochs: reaches alpha_end by decay_epochs
        frac = min(epoch / self.decay_epochs, 1.0) ** self.decay_power
        self.alpha = self.alpha_start - (self.alpha_start - self.alpha_end) * frac
        self.alpha = max(self.alpha_end, self.alpha)
        return self.alpha

    def forward(
        self,
        logits: torch.Tensor,
        target: torch.Tensor,
        return_components: bool = False,
    ) -> torch.Tensor | Tuple[torch.Tensor, Dict[str, float]]:
        # Ensure pos_weight is on the same device as logits
        if self.bce.pos_weight is not None and self.bce.pos_weight.device != logits.device:
            self.bce.pos_weight = self.bce.pos_weight.to(logits.device)

        # Force float32 computation for numerical stability and to prevent autocast issues.
        with torch.amp.autocast(device_type=logits.device.type, enabled=False):
            logits_f32 = logits.float()
            target_f32 = target.float()
            bce_loss = self.bce(logits_f32, target_f32)
            dice_loss = self.dice(logits_f32, target_f32)
            cldice_loss = self.cldice(logits_f32, target_f32)

        cldice_w = max(0.0, 1.0 - self.alpha - self.dice_weight)
        total_loss = self.alpha * bce_loss + self.dice_weight * dice_loss + cldice_w * cldice_loss

        if return_components:
            components = {
                "total_loss": total_loss.item(),
                "bce_loss": bce_loss.item(),
                "dice_loss": dice_loss.item(),
                "cldice_loss": cldice_loss.item(),
                "alpha": self.alpha,
            }
            return total_loss, components

        return total_loss


# =====================================================================
#  Fragmentation Metric (Connected Component Ratio)
# =====================================================================

def avg_component_count(pred_masks: torch.Tensor, threshold: float = 0.5) -> float:
    """
    Computes average number of connected components in predictions.
    Lower count indicates continuous road networks; high count indicates fragmentation.
    """
    from scipy import ndimage
    counts = []
    preds_np = (torch.sigmoid(pred_masks) > threshold).cpu().numpy().astype("uint8")
    for m in preds_np:
        if m.ndim == 3:
            m = m[0]
        _, n = ndimage.label(m)
        counts.append(n)
    return float(sum(counts) / max(len(counts), 1))
# ---- end backend/src/utils/loss.py ----

# `style` module built in-memory from scripts/paper_figures/style.py
style = types.ModuleType('style')
exec(compile("\"\"\"\nShared figure style for every paper figure (local and Kaggle-generated).\n\nPalette: the dataviz reference categorical palette, validated on a white surface\n(scripts/validate_palette.js: all hard gates pass; aqua/yellow sit below 3:1\ncontrast, so bars in those colors always carry visible value labels).\nColors follow the model, never its rank, in every figure.\n\"\"\"\n\nimport matplotlib as mpl\nimport matplotlib.pyplot as plt\n\nINK = \"#0b0b0b\"\nINK_2 = \"#52514e\"\nMUTED = \"#898781\"\nGRID = \"#e1e0d9\"\nAXIS = \"#c3c2b7\"\nSURFACE = \"#ffffff\"\n\nBLUE, ORANGE, AQUA, YELLOW = \"#2a78d6\", \"#eb6834\", \"#1baf7a\", \"#eda100\"\nCONTEXT_GRAY = \"#b9b7ae\"   # non-highlighted reference bars\n\n# Entity -> color, fixed across all figures.\nMODEL_COLORS = {\n    \"final\": BLUE,        # v2, run 2, epoch 53 (best_model_v2.pth)\n    \"baseline\": ORANGE,   # June baseline, no attention gates (best_model_new.pth)\n    \"run1\": AQUA,         # v2, run 1, epoch 18\n    \"collapsed\": YELLOW,  # v2, collapsed run, epoch 46\n}\nMODEL_LABELS = {\n    \"final\": \"Proposed (v2, final)\",\n    \"baseline\": \"Baseline (no gates)\",\n    \"run1\": \"v2, run 1 (ep. 18)\",\n    \"collapsed\": \"v2, collapsed (ep. 46)\",\n}\n\n# Error maps: correct pixels recede to neutral ink so errors carry the color.\nTP_COLOR, FP_COLOR, FN_COLOR = \"#52514e\", ORANGE, BLUE\n\n# Paper widths (IEEE): single column 3.5 in, double column 7.16 in.\nCOL_W, PAGE_W = 3.5, 7.16\n\n\ndef apply():\n    mpl.rcParams.update({\n        \"font.family\": \"sans-serif\",\n        \"font.sans-serif\": [\"Arial\", \"DejaVu Sans\", \"Liberation Sans\"],\n        \"font.size\": 8,\n        \"axes.titlesize\": 8.5,\n        \"axes.titleweight\": \"bold\",\n        \"axes.labelsize\": 8,\n        \"axes.labelcolor\": INK_2,\n        \"axes.edgecolor\": AXIS,\n        \"axes.linewidth\": 0.6,\n        \"axes.facecolor\": SURFACE,\n        \"axes.grid\": True,\n        \"axes.axisbelow\": True,\n        \"axes.spines.top\": False,\n        \"axes.spines.right\": False,\n        \"grid.color\": GRID,\n        \"grid.linewidth\": 0.5,\n        \"xtick.color\": MUTED,\n        \"ytick.color\": MUTED,\n        \"xtick.labelcolor\": INK_2,\n        \"ytick.labelcolor\": INK_2,\n        \"xtick.labelsize\": 7.5,\n        \"ytick.labelsize\": 7.5,\n        \"xtick.major.size\": 2.5,\n        \"ytick.major.size\": 2.5,\n        \"legend.fontsize\": 7.5,\n        \"legend.frameon\": False,\n        \"lines.linewidth\": 1.5,\n        \"figure.facecolor\": SURFACE,\n        \"savefig.facecolor\": SURFACE,\n        \"savefig.dpi\": 300,\n        \"savefig.bbox\": \"tight\",\n        \"savefig.pad_inches\": 0.03,\n        \"pdf.fonttype\": 42,   # embed TrueType -- required by most IEEE/Overleaf checks\n        \"ps.fonttype\": 42,\n    })\n\n\ndef save(fig, out_dir, name):\n    \"\"\"Writes PNG (300 DPI, for slides/Word) and PDF (vector, for LaTeX).\"\"\"\n    import os\n    os.makedirs(out_dir, exist_ok=True)\n    for ext in (\"png\", \"pdf\"):\n        fig.savefig(os.path.join(out_dir, f\"{name}.{ext}\"))\n    plt.close(fig)\n\n\ndef panel_label(ax, letter):\n    ax.text(-0.02, 1.02, f\"({letter})\", transform=ax.transAxes, ha=\"right\", va=\"bottom\",\n            fontsize=8.5, fontweight=\"bold\", color=INK)\n\n\ndef image_axes(ax):\n    ax.set_xticks([]); ax.set_yticks([])\n    ax.grid(False)\n    for s in ax.spines.values():\n        s.set_visible(False)\n", 'scripts/paper_figures/style.py', 'exec'), style.__dict__)

In [ ]:
# ── 4. Checkpoints and training logs attached as inputs ──
ROLE_BY_FILENAME = {
    "best_model_v2.pth": "final",
    "best_model_v2_new.pth": "final",
    "best_model_v2_epoch18_iou0159.pth": "run1",
    "best_model_v2_collapsed_epoch46.pth": "collapsed",
    "best_model_new.pth": "baseline",
}
ROLE_ORDER = ["baseline", "collapsed", "run1", "final"]


def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


CHECKPOINTS = {}
for path in sorted(p for root in INPUT_ROOTS for p in glob.glob(os.path.join(root, "**", "*.pth"), recursive=True)):
    role = ROLE_BY_FILENAME.get(os.path.basename(path))
    if role is None:
        print("ignored (unrecognized name):", path)
        continue
    digest = md5(path)
    if role in CHECKPOINTS:
        if CHECKPOINTS[role]["md5"] != digest:
            print(f"WARNING: two different files for role '{role}'; keeping {CHECKPOINTS[role]['path']}")
        continue
    _, meta = load_mobilevit_checkpoint(path, device="cpu")
    CHECKPOINTS[role] = {"path": path, "md5": digest,
                         "meta": {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in meta.items()}}
ROLES = [r for r in ROLE_ORDER if r in CHECKPOINTS]
assert "final" in ROLES, "Attach a dataset containing best_model_v2.pth (the final checkpoint)."
for r in ROLES:
    print(f"{r:10s} {CHECKPOINTS[r]['path']}  stored={CHECKPOINTS[r]['meta']}")

TRAINING_LOGS = []
for path in sorted(p for root in INPUT_ROOTS
                   for p in glob.glob(os.path.join(root, "**", "wandb_local*.json"), recursive=True)):
    try:
        records = json.load(open(path))
        epochs = sorted([r for r in records if "val/epoch_loss" in r], key=lambda r: r["epoch"])
        if epochs:
            TRAINING_LOGS.append({"path": path, "epochs": epochs, "has_iou": "val/iou" in epochs[0]})
            print(f"training log: {path} ({len(epochs)} epochs, val IoU logged: {'val/iou' in epochs[0]})")
    except Exception as e:
        print("could not read", path, e)

In [ ]:
# ── 5. Reproduce the training split exactly ──
def list_ids(img_dir, mask_dir):
    """Identical to the training notebooks' DeepGlobeDataset id listing (directory order)."""
    ids = []
    for f in os.listdir(img_dir):
        if f.endswith(".jpg"):
            tid = f.split("_")[0]
            if os.path.exists(os.path.join(mask_dir, f"{tid}_mask.png")):
                ids.append(tid)
    return ids


ALL_IDS = list_ids(TRAIN_DIR, TRAIN_DIR)
_shuffled = list(ALL_IDS)
random.Random(SEED).shuffle(_shuffled)
VAL_IDS = _shuffled[: max(1, int(len(_shuffled) * 0.1))]
EVAL_IDS = VAL_IDS[:MAX_TILES] if MAX_TILES else VAL_IDS
SPLIT = {
    "n_labeled": len(ALL_IDS), "n_val": len(VAL_IDS), "n_eval": len(EVAL_IDS),
    "listing_order_sha1": hashlib.sha1("\n".join(ALL_IDS).encode()).hexdigest(),
    "val_ids_sha1": hashlib.sha1("\n".join(VAL_IDS).encode()).hexdigest(),
}
json.dump({"val_ids": VAL_IDS, **SPLIT}, open(os.path.join(OUT_DIR, "val_ids.json"), "w"), indent=1)
print(SPLIT, "| first val ids:", VAL_IDS[:5])

In [ ]:
# ── 6. Verify the split: re-run the training-time validation protocol ──
# Training validated on tiles RESIZED to 256x256, batch 16, AMP, and stored the resulting
# metrics in each checkpoint. Reproducing them to within TOL on these tiles is strong
# evidence the split is the same, i.e. no validation tile was ever trained on.
TOL = 2e-3
IMNET_MEAN, IMNET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class TrainingProtocolDS(Dataset):
    tf = A.Compose([A.Resize(height=256, width=256),
                    A.Normalize(mean=IMNET_MEAN, std=IMNET_STD, max_pixel_value=255.0), ToTensorV2()])

    def __init__(self, ids):
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        tid = self.ids[i]
        image = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_DIR, f"{tid}_sat.jpg")), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(os.path.join(TRAIN_DIR, f"{tid}_mask.png"), cv2.IMREAD_GRAYSCALE)
        out = self.tf(image=image, mask=mask)
        m = out["mask"]
        m = (m.to(torch.float32) if torch.is_tensor(m) else torch.tensor(m, dtype=torch.float32)) / 255.0
        return out["image"], m.unsqueeze(0)


def soft_cldice_per_image(logits_f32, target, num_iter=10, smooth=1.0):
    """SoftClDiceLoss's score, returned per image instead of batch-averaged."""
    p = torch.sigmoid(logits_f32)
    sp, st = soft_skel(p, num_iter), soft_skel(target, num_iter)
    tprec = ((sp * target).sum((1, 2, 3)) + smooth) / (sp.sum((1, 2, 3)) + smooth)
    tsens = ((st * p).sum((1, 2, 3)) + smooth) / (st.sum((1, 2, 3)) + smooth)
    return 2.0 * tprec * tsens / (tprec + tsens + 1e-7)


@torch.no_grad()
def training_protocol_eval(model, ids):
    loader = DataLoader(TrainingProtocolDS(ids), batch_size=16, shuffle=False, num_workers=NUM_WORKERS)
    cl = SoftClDiceLoss(num_iter=10, smooth=1.0)
    sums = {"iou": 0.0, "precision": 0.0, "positive_frac": 0.0, "cldice": 0.0}
    per_img_iou, per_img_cl, n_batches = [], [], 0
    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            preds = model(images)
        sums["cldice"] += cl(preds.float(), masks.float()).item()     # training computed this in fp32
        per_img_cl.append(soft_cldice_per_image(preds.float(), masks.float()).cpu())
        pb = (torch.sigmoid(preds) > 0.5).float()                     # training thresholded the AMP output
        tb = (masks > 0.5).float()
        inter = (pb * tb).sum((1, 2, 3))
        union = (pb + tb).clamp(0, 1).sum((1, 2, 3))
        iou = (inter + 1e-7) / (union + 1e-7)
        sums["iou"] += iou.mean().item()
        sums["precision"] += ((inter + 1e-7) / (pb.sum((1, 2, 3)) + 1e-7)).mean().item()
        sums["positive_frac"] += pb.mean().item()
        per_img_iou.append(iou.cpu())
        n_batches += 1
    res = {k: v / n_batches for k, v in sums.items()}
    return res, torch.cat(per_img_iou).numpy(), torch.cat(per_img_cl).numpy()


VERIFY, TRAIN_PROTOCOL, TRAIN_PROTOCOL_PER_TILE = {}, {}, {}
STORED_KEYS = {"val_iou": "iou", "val_precision": "precision", "val_positive_frac": "positive_frac", "val_cldice": "cldice"}
for role in ROLES:
    model, _ = load_mobilevit_checkpoint(CHECKPOINTS[role]["path"], device=DEVICE)
    res, per_iou, per_cl = training_protocol_eval(model, VAL_IDS)   # always the full split: stored values cover it
    res["cldice_score"] = 1.0 - res["cldice"]
    TRAIN_PROTOCOL[role] = res
    TRAIN_PROTOCOL_PER_TILE[role] = {"iou": per_iou, "soft_cldice": per_cl}
    checks = {}
    for stored_key, ours in STORED_KEYS.items():
        if stored_key in CHECKPOINTS[role]["meta"]:
            stored = CHECKPOINTS[role]["meta"][stored_key]
            checks[stored_key] = {"stored": stored, "recomputed": res[ours], "abs_diff": abs(stored - res[ours]),
                                  "match": abs(stored - res[ours]) <= TOL}
    VERIFY[role] = checks
    del model
    torch.cuda.empty_cache()
    print(f"\n{role}: recomputed training-protocol metrics {({k: round(v, 5) for k, v in res.items()})}")
    for k, c in checks.items():
        print(f"   {k:18s} stored={c['stored']:.5f} recomputed={c['recomputed']:.5f} "
              f"diff={c['abs_diff']:.5f} {'MATCH' if c['match'] else 'MISMATCH'}")

_v2_checks = [c for r in ("final", "run1") if r in VERIFY for c in VERIFY[r].values()]
SPLIT_VERIFIED = bool(_v2_checks) and all(c["match"] for c in _v2_checks)
print("\nSPLIT VERIFIED" if SPLIT_VERIFIED else
      "\n*** SPLIT NOT VERIFIED — stored metrics were not reproduced; these tiles may include training "
      "tiles, so the metrics below would be optimistic. Do not report them as held-out results. ***")

In [ ]:
# ── 7. Canopy (vegetation) mask: Excess-Green index + Otsu threshold ──
# Road pixels whose image colour is vegetation-like are treated as canopy-occluded.
# ExG = 2g - r - b on chromatic coordinates (Woebbecke et al., 1995); the threshold is
# Otsu's on pixels sampled from the evaluated tiles -- data-driven, not hand-tuned.
def exg(rgb):
    rgb = rgb.astype(np.float32)
    s = rgb.sum(axis=2) + 1e-6
    return (2 * rgb[..., 1] - rgb[..., 0] - rgb[..., 2]) / s


def load_tile(tid):
    rgb = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_DIR, f"{tid}_sat.jpg")), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(os.path.join(TRAIN_DIR, f"{tid}_mask.png"), cv2.IMREAD_GRAYSCALE) > 127
    return rgb, gt


_rng = np.random.default_rng(0)
_samples = []
for tid in EVAL_IDS:
    e = exg(load_tile(tid)[0]).ravel()
    _samples.append(e[_rng.integers(0, e.size, 4000)])
VEG_T = float(threshold_otsu(np.concatenate(_samples)))
print(f"ExG Otsu threshold = {VEG_T:.4f}")

In [ ]:
# ── 8. Native-resolution evaluation ──
RHO = 3   # relaxed-matching tolerance in pixels (1.5 m at 0.5 m/px), after Mnih & Hinton (2010)
# Which binarizations to evaluate, per checkpoint and probability source.
#   raw        : probability > 0.5
#   hyst       : hysteresis thresholding (0.35 / 0.12)
#   hyst_close : + 5x5 morphological closing
#   full_orig  : + canopy-gap bridging as originally written (also bridges tile-border exits)
#   full       : + canopy-gap bridging with the border fix  (the deployed post-processing)
# Sources: "single" = one forward pass, "tta" = mean of 4 flips (the deployed inference).
VARIANTS = {
    "final":     {"single": ["raw", "hyst", "hyst_close", "full_orig", "full"], "tta": ["raw", "full_orig", "full"]},
    "baseline":  {"single": ["raw", "full"], "tta": ["full"]},
    "run1":      {"single": ["raw"], "tta": ["full"]},
    "collapsed": {"single": ["raw"], "tta": []},
}
COUNT_KEYS = ["tp", "fp", "fn", "npred", "rel_tp_pred", "rel_tp_gt", "sk_p", "sk_p_in_g", "sk_g", "sk_g_in_p",
              "can_gt", "can_hit", "can_hit_rel", "open_gt", "open_hit", "open_hit_rel"]


class NativeDS(Dataset):
    def __init__(self, ids):
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        rgb = cv2.cvtColor(cv2.imread(os.path.join(TRAIN_DIR, f"{self.ids[i]}_sat.jpg")), cv2.COLOR_BGR2RGB)
        x = (rgb.astype(np.float32) / 255.0 - np.array(IMNET_MEAN, np.float32)) / np.array(IMNET_STD, np.float32)
        return i, torch.from_numpy(x.transpose(2, 0, 1))


@torch.no_grad()
def forward_probs(model, x, tta):
    with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
        p = torch.sigmoid(model(x)).float()
        if not tta:
            return p, None
        acc = p.clone()
        for dims in ([3], [2], [2, 3]):
            acc += torch.flip(torch.sigmoid(model(torch.flip(x, dims))).float(), dims)
    return p, acc / 4.0


def binarize_all(probs, variants):
    out = {}
    if "raw" in variants:
        out["raw"] = probs > 0.5
    if any(v in variants for v in ("hyst", "hyst_close", "full_orig", "full")):
        hyst = hysteresis_threshold(probs, high_thresh=0.35, low_thresh=0.12)
        if "hyst" in variants:
            out["hyst"] = hyst > 0
        closed = cv2.morphologyEx(hyst, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8), iterations=1)
        if "hyst_close" in variants:
            out["hyst_close"] = closed > 0
        if "full_orig" in variants:
            out["full_orig"] = connect_canopy_gaps(closed, max_gap_dist=220.0, max_angle_deg=65.0,
                                                   road_width=6, border_margin=0) > 0
        if "full" in variants:
            out["full"] = connect_canopy_gaps(closed, max_gap_dist=220.0, max_angle_deg=65.0,
                                              road_width=6, border_margin=16) > 0
    return out


def count_metrics(P, G, V, near_g, sk_g):
    if P.any():
        near_p = cv2.distanceTransform((~P).astype(np.uint8), cv2.DIST_L2, 3) <= RHO
    else:
        near_p = np.zeros_like(P)
    sk_p = skeletonize(P)
    gv, go = G & V, G & ~V
    return {
        "tp": int((P & G).sum()), "fp": int((P & ~G).sum()), "fn": int((~P & G).sum()), "npred": int(P.sum()),
        "rel_tp_pred": int((P & near_g).sum()), "rel_tp_gt": int((G & near_p).sum()),
        "sk_p": int(sk_p.sum()), "sk_p_in_g": int((sk_p & G).sum()),
        "sk_g": int(sk_g.sum()), "sk_g_in_p": int((sk_g & P).sum()),
        "cc_pred": int(cv2.connectedComponents(P.astype(np.uint8), connectivity=8)[0] - 1),
        "can_gt": int(gv.sum()), "can_hit": int((gv & P).sum()), "can_hit_rel": int((gv & near_p).sum()),
        "open_gt": int(go.sum()), "open_hit": int((go & P).sum()), "open_hit_rel": int((go & near_p).sum()),
    }


def tile_worker(job):
    tid, probs_by_src, variants = job
    rgb, G = load_tile(tid)
    V = exg(rgb) > VEG_T
    near_g = (cv2.distanceTransform((~G).astype(np.uint8), cv2.DIST_L2, 3) <= RHO) if G.any() else np.zeros_like(G)
    sk_g = skeletonize(G)
    rec = {"tile": tid, "gt_pixels": int(G.sum()), "canopy_ratio": float((G & V).sum() / max(int(G.sum()), 1)),
           "cc_gt": int(cv2.connectedComponents(G.astype(np.uint8), connectivity=8)[0] - 1),
           "pixels": int(G.size), "variants": {}, "hist": {}}
    for src, vlist in variants.items():
        if not vlist and src != "single":
            continue
        probs = probs_by_src[src].astype(np.float32)
        q = np.clip(np.rint(probs * 255), 0, 255).astype(np.int64)
        rec["hist"][src] = (np.bincount(q[G], minlength=256), np.bincount(q[~G], minlength=256))
        for v, P in binarize_all(probs, vlist).items():
            rec["variants"][f"{src}_{v}"] = count_metrics(P, G, V, near_g, sk_g)
    return rec


def _pool():
    import multiprocessing as mp
    if "fork" in mp.get_all_start_methods():
        return mp.get_context("fork").Pool(os.cpu_count())
    return None   # e.g. Windows smoke tests: run serially


RECORDS = {}        # role -> list of per-tile records
HIST = {}           # role -> src -> (pos_hist, neg_hist)
t_start = time.time()
pool = _pool()
for role in ROLES:
    model, _ = load_mobilevit_checkpoint(CHECKPOINTS[role]["path"], device=DEVICE)
    need_tta = bool(VARIANTS[role]["tta"])
    loader = DataLoader(NativeDS(EVAL_IDS), batch_size=4 if DEVICE.type == "cuda" else 1,
                        shuffle=False, num_workers=NUM_WORKERS)
    RECORDS[role], HIST[role], jobs = [], {}, []

    def _flush(jobs):
        results = pool.map(tile_worker, jobs) if pool else [tile_worker(j) for j in jobs]
        for rec in results:
            for src, (hp, hn) in rec.pop("hist").items():
                a = HIST[role].setdefault(src, [np.zeros(256, np.int64), np.zeros(256, np.int64)])
                a[0] += hp; a[1] += hn
            RECORDS[role].append(rec)

    for idx, x in loader:
        p, pt = forward_probs(model, x.to(DEVICE), need_tta)
        for k, i in enumerate(idx.tolist()):
            srcs = {"single": p[k, 0].cpu().numpy().astype(np.float16)}
            if pt is not None:
                srcs["tta"] = pt[k, 0].cpu().numpy().astype(np.float16)
            jobs.append((EVAL_IDS[i], srcs, VARIANTS[role]))
        if len(jobs) >= 32:
            _flush(jobs); jobs = []
    if jobs:
        _flush(jobs)
    del model
    torch.cuda.empty_cache()
    print(f"{role}: {len(RECORDS[role])} tiles evaluated ({(time.time() - t_start) / 60:.1f} min elapsed)")
if pool:
    pool.close()

In [ ]:
# ── 9. Aggregate: dataset-level metrics, bootstrap CIs, paired tests ──
def derive(s):
    div = lambda a, b: a / b if b else float("nan")
    gt = s["tp"] + s["fn"]
    iou = div(s["tp"], s["tp"] + s["fp"] + s["fn"])
    p, r = div(s["tp"], s["tp"] + s["fp"]), div(s["tp"], gt)
    rp, rr = div(s["rel_tp_pred"], s["npred"]), div(s["rel_tp_gt"], gt)
    tprec, tsens = div(s["sk_p_in_g"], s["sk_p"]), div(s["sk_g_in_p"], s["sk_g"])
    f = lambda a, b: div(2 * a * b, a + b)
    return {"iou": iou, "precision": p, "recall": r, "f1": f(p, r),
            "relaxed_precision": rp, "relaxed_recall": rr, "relaxed_f1": f(rp, rr),
            "cldice": f(tprec, tsens), "topo_precision": tprec, "topo_sensitivity": tsens,
            "canopy_recall": div(s["can_hit"], s["can_gt"]), "canopy_recall_relaxed": div(s["can_hit_rel"], s["can_gt"]),
            "open_recall": div(s["open_hit"], s["open_gt"]), "open_recall_relaxed": div(s["open_hit_rel"], s["open_gt"])}


def count_matrix(role, variant):
    recs = RECORDS[role]
    return np.array([[r["variants"][variant][k] for k in COUNT_KEYS] for r in recs], dtype=np.float64)


def summarize(mat):
    return derive(dict(zip(COUNT_KEYS, mat.sum(0))))


def bootstrap_ci(mat, keys=("iou", "f1", "relaxed_f1", "cldice"), reps=1000, seed=0):
    rng = np.random.default_rng(seed)
    n = mat.shape[0]
    draws = {k: [] for k in keys}
    for _ in range(reps):
        d = summarize(mat[rng.integers(0, n, n)])
        for k in keys:
            draws[k].append(d[k])
    return {k: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))] for k, v in draws.items()}


def per_tile_table(role, variant):
    rows = []
    for r in RECORDS[role]:
        c = r["variants"][variant]
        d = derive(c)
        union = c["tp"] + c["fp"] + c["fn"]
        rows.append({"tile": r["tile"], "model": role, "variant": variant,
                     "iou": d["iou"] if union else np.nan, "precision": d["precision"], "recall": d["recall"],
                     "f1": d["f1"], "relaxed_f1": d["relaxed_f1"], "cldice": d["cldice"],
                     "components_pred": c["cc_pred"], "components_gt": r["cc_gt"],
                     "positive_frac": c["npred"] / r["pixels"], "gt_frac": r["gt_pixels"] / r["pixels"],
                     "canopy_ratio": r["canopy_ratio"], "gt_pixels": r["gt_pixels"]})
    return pd.DataFrame(rows)


CONFIGS = []   # (role, variant) pairs actually evaluated, in display order
for role in ROLES:
    for src in ("single", "tta"):
        for v in VARIANTS[role][src]:
            CONFIGS.append((role, f"{src}_{v}"))

RESULTS = {}
PER_TILE = pd.concat([per_tile_table(r, v) for r, v in CONFIGS], ignore_index=True)
for role, variant in CONFIGS:
    mat = count_matrix(role, variant)
    pt = PER_TILE[(PER_TILE.model == role) & (PER_TILE.variant == variant)]
    RESULTS.setdefault(role, {})[variant] = {
        **summarize(mat), "ci95": bootstrap_ci(mat),
        "mean_tile_iou": float(np.nanmean(pt.iou)), "median_components_per_tile": float(np.median(pt.components_pred)),
        "median_gt_components_per_tile": float(np.median(pt.components_gt)),
        "mean_positive_frac": float(pt.positive_frac.mean()), "mean_gt_frac": float(pt.gt_frac.mean()),
    }

DEPLOYED = "tta_full"


def paired(role_a, var_a, role_b, var_b, metric="iou"):
    a = PER_TILE[(PER_TILE.model == role_a) & (PER_TILE.variant == var_a)].set_index("tile")[metric]
    b = PER_TILE[(PER_TILE.model == role_b) & (PER_TILE.variant == var_b)].set_index("tile")[metric]
    df = pd.concat([a, b], axis=1, keys=["a", "b"]).dropna()
    diff = df.a - df.b
    w = sstats.wilcoxon(df.a, df.b) if len(df) > 10 and (diff != 0).any() else None
    return {"n_tiles": int(len(df)), "mean_diff": float(diff.mean()), "median_diff": float(diff.median()),
            "frac_tiles_improved": float((diff > 0).mean()), "wilcoxon_p": float(w.pvalue) if w else None}


TESTS = {}
if "baseline" in ROLES:
    TESTS["final_vs_baseline_deployed_iou"] = paired("final", DEPLOYED, "baseline", DEPLOYED)
    TESTS["final_vs_baseline_single_raw_iou"] = paired("final", "single_raw", "baseline", "single_raw")
    TESTS["final_vs_baseline_deployed_cldice"] = paired("final", DEPLOYED, "baseline", DEPLOYED, "cldice")
TESTS["final_tta_vs_single_full_iou"] = paired("final", "tta_full", "final", "single_full")

# Canopy tertiles over tiles that actually contain road.
_road = PER_TILE[(PER_TILE.model == "final") & (PER_TILE.variant == DEPLOYED) & (PER_TILE.gt_pixels >= 1000)]
TERTILE_EDGES = np.quantile(_road.canopy_ratio, [0, 1 / 3, 2 / 3, 1]).tolist()
TILE_TERTILE = {t: int(np.clip(np.searchsorted(TERTILE_EDGES[1:-1], c, side="right"), 0, 2))
                for t, c in zip(_road.tile, _road.canopy_ratio)}
CANOPY_TERTILES = {}
for role, variant in CONFIGS:
    if variant not in ("single_raw", DEPLOYED):
        continue
    ids = [r["tile"] for r in RECORDS[role]]
    mat = count_matrix(role, variant)
    for t in range(3):
        sel = np.array([TILE_TERTILE.get(i) == t for i in ids])
        if sel.sum() >= 5:
            CANOPY_TERTILES.setdefault(f"{role}/{variant}", {})[t] = {
                **{k: v for k, v in summarize(mat[sel]).items() if k in ("iou", "relaxed_f1", "cldice", "recall")},
                "ci95": bootstrap_ci(mat[sel], keys=("relaxed_f1", "cldice")), "n_tiles": int(sel.sum())}

# Threshold sweeps from the pooled probability histograms (exact at 1/255 resolution).
SWEEPS = {}
for role in ROLES:
    for src, (hp, hn) in HIST[role].items():
        tp = hp[::-1].cumsum()[::-1].astype(np.float64)   # pixels with q >= t
        fp = hn[::-1].cumsum()[::-1].astype(np.float64)
        pos = float(hp.sum())
        th = np.arange(256) / 255.0
        with np.errstate(divide="ignore", invalid="ignore"):
            prec = np.where(tp + fp > 0, tp / (tp + fp), np.nan)
            rec = tp / pos
            iou = tp / (fp + pos)
            f1 = 2 * prec * rec / (prec + rec)
        SWEEPS[f"{role}/{src}"] = {"threshold": th.tolist(), "precision": prec.tolist(), "recall": rec.tolist(),
                                   "iou": iou.tolist(), "f1": f1.tolist()}

print(json.dumps({r: {v: {k: round(x, 4) for k, x in m.items() if isinstance(x, float)}
                      for v, m in d.items()} for r, d in RESULTS.items()}, indent=1)[:4000])
print(json.dumps(TESTS, indent=1))

In [ ]:
# ── 10. Save results: JSON, per-tile CSV, LaTeX table ──
NAMES = {"baseline": "Baseline (June, no attention gates)", "collapsed": "v2, collapsed run (ep. 46)",
         "run1": "v2, run 1 (ep. 18)", "final": "Proposed v2 (ep. 53)"}
VARIANT_NAMES = {"single_raw": "single pass, p > 0.5", "single_hyst": "hysteresis",
                 "single_hyst_close": "hysteresis + closing", "single_full_orig": "+ gap bridging (original)",
                 "single_full": "+ gap bridging (border fix)", "tta_raw": "TTA, p > 0.5",
                 "tta_full_orig": "TTA + post-proc. (original bridging)",
                 "tta_full": "TTA + post-proc. (deployed)"}

OUT = {
    "env": ENV, "split": SPLIT, "split_verified": SPLIT_VERIFIED, "verification": VERIFY,
    "training_protocol_metrics": TRAIN_PROTOCOL, "checkpoints": CHECKPOINTS,
    "native_resolution": RESULTS, "paired_tests": TESTS,
    "canopy": {"exg_otsu_threshold": VEG_T, "tertile_edges": TERTILE_EDGES, "by_tertile": CANOPY_TERTILES},
    "threshold_sweeps": SWEEPS, "relaxed_tolerance_px": RHO,
    "protocol": {"resolution": "native 1024x1024 tiles", "threshold": 0.5,
                 "iou": "dataset-level (pixels pooled over all tiles), road class",
                 "clDice": "hard, skimage skeletonize, pooled counts",
                 "ci": "95% percentile bootstrap over tiles, 1000 resamples"},
}
json.dump(OUT, open(os.path.join(OUT_DIR, "results.json"), "w"), indent=1, default=float)
PER_TILE.to_csv(os.path.join(OUT_DIR, "per_tile_metrics.csv"), index=False)

pct = lambda v: f"{100 * v:.1f}" if v == v else "--"
rows = []
for role, variant in CONFIGS:
    if variant not in ("single_raw", DEPLOYED):
        continue
    m = RESULTS[role][variant]
    ci = m["ci95"]["iou"]
    rows.append(f"{NAMES[role]} & {VARIANT_NAMES[variant]} & {pct(m['iou'])} [{pct(ci[0])}, {pct(ci[1])}] & "
                f"{pct(m['precision'])} & {pct(m['recall'])} & {pct(m['f1'])} & {pct(m['relaxed_f1'])} & "
                f"{pct(m['cldice'])} & {m['median_components_per_tile']:.0f} \\\\")
latex = (
    "% Generated by notebooks/evaluate_for_paper.ipynb -- measured values, do not edit by hand.\n"
    f"% Split verified against checkpoint-stored metrics: {SPLIT_VERIFIED}. Tiles: {SPLIT['n_eval']}.\n"
    "\\begin{table*}[t]\n\\centering\n"
    f"\\caption{{Road extraction on the DeepGlobe validation split ({SPLIT['n_eval']} tiles, native "
    "$1024\\times1024$ resolution). Dataset-level pixel metrics (\\%); IoU with 95\\% bootstrap CI; "
    f"relaxed F1 uses a $\\rho={RHO}$\\,px tolerance; clDice on hard skeletons.}}\n"
    "\\label{tab:deepglobe_results}\n\\small\n"
    "\\begin{tabular}{llccccccc}\n\\toprule\n"
    "Model & Inference & IoU & Prec. & Rec. & F1 & Relaxed F1 & clDice & Comp./tile \\\\\n\\midrule\n"
    + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}\n\\end{table*}\n")
open(os.path.join(OUT_DIR, "results_table.tex"), "w").write(latex)
print(latex)

In [ ]:
# ── 11. Figures ──
style.apply()
C, L = style.MODEL_COLORS, style.MODEL_LABELS
SPLIT_NOTE = "" if SPLIT_VERIFIED else "  [split NOT verified]"
FOOT = f"DeepGlobe validation split, {SPLIT['n_eval']} tiles, native 1024\u00b2 resolution{SPLIT_NOTE}"


def footnote(fig, text=FOOT, y=None):
    """Placed below everything already drawn (tick labels, x-labels, legends), so it never collides."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bottom = min(a.get_tightbbox(renderer).y0 for a in fig.axes)
    for leg in fig.legends:
        bottom = min(bottom, leg.get_window_extent(renderer).y0)
    y_auto = bottom / fig.bbox.height - 0.025
    fig.text(0.0, min(y_auto, y) if y is not None else y_auto, text, fontsize=6.5, color=style.MUTED, va="top")


def hbars(ax, labels, values, colors, cis=None, fmt="{:.1f}", hatches=None, xmax=None):
    y = np.arange(len(labels))[::-1]
    ax.barh(y, values, height=0.64, color=colors, edgecolor="white", linewidth=1.0,
            hatch=None if hatches is None else None)
    if hatches:
        for yi, v, h in zip(y, values, hatches):
            if h:
                ax.barh(yi, v, height=0.64, color="none", edgecolor="white", linewidth=0, hatch=h)
    if cis:
        for yi, (lo, hi) in zip(y, cis):
            ax.plot([lo, hi], [yi, yi], color=style.INK, lw=0.8, solid_capstyle="butt")
    reach = max((ci[1] if ci else v) for v, ci in zip(values, cis or [None] * len(values)) if v == v)
    top = xmax or reach * 1.22
    for yi, v, ci in zip(y, values, cis or [None] * len(values)):
        xpos = (ci[1] if ci else v) + top * 0.015
        ax.text(xpos, yi, fmt.format(v), va="center", fontsize=6.8, color=style.INK_2)
    ax.set_xlim(0, top)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.grid(axis="y", visible=False)
    return y


# 11a. Validation protocol: 256-resize (training-time) vs native resolution
fig, ax = plt.subplots(figsize=(style.COL_W, 0.42 * len(ROLES) + 0.75))
y = np.arange(len(ROLES))[::-1]
for yi, role in zip(y, ROLES):
    a = 100 * TRAIN_PROTOCOL[role]["iou"]
    b = 100 * RESULTS[role]["single_raw"]["iou"]
    ax.plot([a, b], [yi, yi], color=style.AXIS, lw=1.5, zorder=1)
    ax.plot(a, yi, "o", ms=6, mfc="white", mec=C[role], mew=1.5, zorder=2)
    ax.plot(b, yi, "o", ms=6, color=C[role], mec="white", mew=1.0, zorder=3)
    ax.text(b + 1.2, yi, f"{b:.1f}", va="center", fontsize=6.8, color=style.INK_2)
    ax.text(a - 1.2, yi, f"{a:.1f}", va="center", ha="right", fontsize=6.8, color=style.MUTED)
ax.set_yticks(y); ax.set_yticklabels([L[r] for r in ROLES]); ax.grid(axis="y", visible=False)
ax.set_xlabel("IoU (%)")
ax.set_xlim(0, max(100 * RESULTS[r]["single_raw"]["iou"] for r in ROLES) * 1.25 + 4)
ax.legend(handles=[Line2D([], [], marker="o", ls="", mfc="white", mec=style.INK_2, label="tiles resized to 256\u00b2 (training-time validation)"),
                   Line2D([], [], marker="o", ls="", color=style.INK_2, label="native 1024\u00b2 (deployment)")],
          loc="lower center", bbox_to_anchor=(0.4, 1.0), ncol=1, fontsize=6.6)
footnote(fig, f"Single forward pass, p > 0.5.{SPLIT_NOTE}", y=-0.08)
style.save(fig, FIG_DIR, "fig_eval_protocol")

# 11b. Main comparison
main_cfgs = [(r, "single_raw") for r in ROLES] + [(r, DEPLOYED) for r in ("baseline", "final") if r in ROLES]
labels = [f"{L[r]}" + (" \u2014 deployed" if v == DEPLOYED else "") for r, v in main_cfgs]
colors = [C[r] for r, _ in main_cfgs]
hatches = ["////" if v == DEPLOYED else None for _, v in main_cfgs]
metrics = [("iou", "IoU"), ("f1", "F1"), ("relaxed_f1", f"Relaxed F1 (\u03c1={RHO}px)"), ("cldice", "clDice")]
fig, axes = plt.subplots(1, 4, figsize=(style.PAGE_W, 0.36 * len(main_cfgs) + 0.9), sharey=True)
xmax = min(118.0, 100 * max(RESULTS[r][v]["ci95"][k][1] for r, v in main_cfgs for k, _ in metrics) * 1.2)
for ax, (k, title), letter in zip(axes, metrics, "abcd"):
    vals = [100 * RESULTS[r][v][k] for r, v in main_cfgs]
    cis = [[100 * c for c in RESULTS[r][v]["ci95"][k]] for r, v in main_cfgs]
    hbars(ax, labels, vals, colors, cis=cis, hatches=hatches, xmax=xmax)
    ax.set_title(title, loc="left"); ax.set_xlabel("%")
    style.panel_label(ax, letter)
footnote(fig, FOOT + ". Hatched = deployed inference (4-flip TTA + hysteresis + closing + gap bridging); "
         "whiskers = 95% bootstrap CI.")
style.save(fig, FIG_DIR, "fig_eval_main")

# 11c. Precision-recall and IoU-vs-threshold
curves = [("baseline", "single", "-"), ("final", "single", "-"), ("final", "tta", "--")]
curves = [c for c in curves if f"{c[0]}/{c[1]}" in SWEEPS]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(style.PAGE_W, 2.45))
for role, src, ls in curves:
    s = SWEEPS[f"{role}/{src}"]
    lab = L[role] + (" + TTA" if src == "tta" else "")
    rec, prec, th = np.array(s["recall"]), np.array(s["precision"]), np.array(s["threshold"])
    ok = ~np.isnan(prec)
    a1.plot(100 * rec[ok], 100 * prec[ok], ls=ls, color=C[role], lw=1.5, label=lab)
    i5 = int(np.argmin(np.abs(th - 0.5)))
    a1.plot(100 * rec[i5], 100 * prec[i5], "o", ms=5, color=C[role], mec="white", mew=1.0)
    iou = 100 * np.array(s["iou"])
    ib = int(np.nanargmax(iou))
    a2.plot(th, iou, ls=ls, color=C[role], lw=1.5, label=f"{lab}: best {iou[ib]:.1f} at {th[ib]:.2f}")
    a2.plot(th[ib], iou[ib], "o", ms=5, color=C[role], mec="white", mew=1.0)
a1.set_xlabel("Recall (%)"); a1.set_ylabel("Precision (%)"); a1.set_xlim(0, 100); a1.set_ylim(0, 100)
a1.set_title("Precision\u2013recall (dot = threshold 0.5)", loc="left")
a2.set_xlabel("Probability threshold"); a2.set_ylabel("IoU (%)"); a2.set_xlim(0, 1)
a2.set_title("IoU vs. threshold (dot = best)", loc="left")
a1.legend(loc="lower left")
a2.legend(loc="upper right", fontsize=6.3)
style.panel_label(a1, "a"); style.panel_label(a2, "b")
footnote(fig, FOOT + ". Pixel-level, pooled over tiles.")
style.save(fig, FIG_DIR, "fig_eval_pr")

# 11d. Post-processing / TTA ablation on the final model
abl = [v for v in ["single_raw", "single_hyst", "single_hyst_close", "single_full_orig", "single_full",
                   "tta_raw", "tta_full_orig", "tta_full"] if v in RESULTS["final"]]
abl_labels = [VARIANT_NAMES[v] for v in abl]
fig, axes = plt.subplots(1, 4, figsize=(style.PAGE_W, 0.34 * len(abl) + 0.9), sharey=True)
panels = [("iou", "IoU (%)", 100, "{:.1f}"), ("relaxed_f1", "Relaxed F1 (%)", 100, "{:.1f}"),
          ("cldice", "clDice (%)", 100, "{:.1f}"), ("median_components_per_tile", "Components / tile (median)", 1, "{:.0f}")]
for ax, (k, title, sc, fmt), letter in zip(axes, panels, "abcd"):
    vals = [sc * RESULTS["final"][v][k] for v in abl]
    cols = [C["final"] if v == DEPLOYED else "#86b6ef" for v in abl]
    cis = [[sc * c for c in RESULTS["final"][v]["ci95"][k]] for v in abl] if k in RESULTS["final"][abl[0]]["ci95"] else None
    hbars(ax, abl_labels, vals, cols, cis=cis, fmt=fmt)
    ax.set_title(title, loc="left")
    style.panel_label(ax, letter)
gt_cc = RESULTS["final"]["single_raw"]["median_gt_components_per_tile"]
axes[3].axvline(gt_cc, color=style.INK_2, lw=0.8, ls=":")
axes[3].text(gt_cc, -0.75, f" ground truth: {gt_cc:.0f}", fontsize=6.5, color=style.INK_2, va="top")
footnote(fig, FOOT + ". Proposed model; dark bar = deployed configuration.")
style.save(fig, FIG_DIR, "fig_eval_ablation")

# 11e. Canopy analysis
can_cfgs = [c for c in [("baseline", "single_raw"), ("final", "single_raw"), ("baseline", DEPLOYED), ("final", DEPLOYED)]
            if c[0] in ROLES]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(style.PAGE_W, 2.5), gridspec_kw={"width_ratios": [1, 1.15]})
groups = [("open_recall_relaxed", "Unoccluded road"), ("canopy_recall_relaxed", "Canopy-covered road")]
w = 0.8 / len(can_cfgs)
for gi, (k, glab) in enumerate(groups):
    for ci_, (role, v) in enumerate(can_cfgs):
        val = 100 * RESULTS[role][v][k]
        x = gi + (ci_ - (len(can_cfgs) - 1) / 2) * w
        a1.bar(x, val, width=w * 0.92, color=C[role], edgecolor="white", linewidth=1.0,
               hatch="////" if v == DEPLOYED else None,
               label=(L[role] + (" \u2014 deployed" if v == DEPLOYED else "")) if gi == 0 else None)
        a1.text(x, val + 1, f"{val:.0f}", ha="center", fontsize=6.3, color=style.INK_2)
a1.set_xticks([0, 1]); a1.set_xticklabels([g for _, g in groups]); a1.grid(axis="x", visible=False)
a1.set_ylabel(f"Relaxed recall (%, \u03c1={RHO}px)"); a1.set_ylim(0, 105)
a1.set_title("Recall of ground-truth road pixels", loc="left")
fig.legend(*a1.get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=4, fontsize=6.4)
tert_labels = ["low", "medium", "high"]
for role in ("baseline", "final"):
    key = f"{role}/{DEPLOYED}"
    if key not in CANOPY_TERTILES:
        continue
    d = CANOPY_TERTILES[key]
    ts = sorted(d)
    vals = [100 * d[t]["cldice"] for t in ts]
    lo = [100 * d[t]["ci95"]["cldice"][0] for t in ts]
    hi = [100 * d[t]["ci95"]["cldice"][1] for t in ts]
    a2.fill_between(ts, lo, hi, color=C[role], alpha=0.12, lw=0)
    a2.plot(ts, vals, "-o", color=C[role], ms=5, mec="white", mew=1.0, label=L[role] + " \u2014 deployed")
    a2.text(ts[-1] + 0.06, vals[-1], f"{vals[-1]:.1f}", va="center", fontsize=6.5, color=style.INK_2)
a2.set_xticks(range(3))
a2.set_xticklabels([f"{n}\n({100*TERTILE_EDGES[i]:.0f}\u2013{100*TERTILE_EDGES[i+1]:.0f}% of road)"
                    for i, n in enumerate(tert_labels)], fontsize=6.6)
a2.set_xlim(-0.3, 2.45); a2.set_ylabel("clDice (%)")
a2.set_title("Connectivity by canopy cover (tertiles)", loc="left")
if a2.lines:
    a2.legend(loc="lower left", fontsize=6.5)
else:
    a2.text(0.5, 0.5, "too few tiles per tertile (< 5)", transform=a2.transAxes, ha="center", color=style.MUTED)
style.panel_label(a1, "a"); style.panel_label(a2, "b")
footnote(fig, FOOT + f". Canopy = ExG > {VEG_T:.3f} (Otsu). Band = 95% bootstrap CI.", y=-0.12)
style.save(fig, FIG_DIR, "fig_eval_canopy")

# 11f. Per-tile distribution and paired difference
if "baseline" in ROLES:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(style.PAGE_W, 2.35), gridspec_kw={"width_ratios": [1, 1.2]})
    dist_cfgs = [("baseline", "single_raw"), ("final", "single_raw"), ("baseline", DEPLOYED), ("final", DEPLOYED)]
    data = [PER_TILE[(PER_TILE.model == r) & (PER_TILE.variant == v)].iou.dropna().values * 100 for r, v in dist_cfgs]
    bp = a1.boxplot(data, vert=False, widths=0.55, patch_artist=True, showfliers=False,
                    medianprops=dict(color=style.INK, lw=1.2), whiskerprops=dict(color=style.AXIS),
                    capprops=dict(color=style.AXIS))
    for patch, (r, v) in zip(bp["boxes"], dist_cfgs):
        patch.set_facecolor(C[r]); patch.set_alpha(0.85); patch.set_edgecolor("white")
        if v == DEPLOYED:
            patch.set_hatch("////")
    a1.set_yticks(range(1, len(dist_cfgs) + 1))
    a1.set_yticklabels([L[r] + (" \u2014 deployed" if v == DEPLOYED else "") for r, v in dist_cfgs])
    a1.set_xlabel("Per-tile IoU (%)"); a1.grid(axis="y", visible=False)
    a1.set_title("Per-tile IoU distribution", loc="left")
    pa = PER_TILE[(PER_TILE.model == "final") & (PER_TILE.variant == DEPLOYED)].set_index("tile").iou
    pb = PER_TILE[(PER_TILE.model == "baseline") & (PER_TILE.variant == DEPLOYED)].set_index("tile").iou
    diff = (pa - pb).dropna() * 100
    bins = np.linspace(-max(abs(diff.min()), abs(diff.max())), max(abs(diff.min()), abs(diff.max())), 41)
    a2.hist(diff[diff <= 0], bins=bins, color=C["baseline"], edgecolor="white", linewidth=0.5, label="baseline better")
    a2.hist(diff[diff > 0], bins=bins, color=C["final"], edgecolor="white", linewidth=0.5, label="proposed better")
    a2.axvline(0, color=style.INK_2, lw=0.8)
    t = TESTS["final_vs_baseline_deployed_iou"]
    ptxt = f"p = {t['wilcoxon_p']:.1e}" if t["wilcoxon_p"] is not None else "p n/a"
    a2.text(0.0, 1.015, f"proposed better on {100*t['frac_tiles_improved']:.0f}% of tiles \u00b7 "
                        f"median \u0394 = {t['median_diff']*100:+.1f} pts \u00b7 Wilcoxon {ptxt}",
            transform=a2.transAxes, ha="left", va="bottom", fontsize=6.6, color=style.INK_2)
    a2.set_xlabel("\u0394 IoU per tile, proposed \u2212 baseline (points)"); a2.set_ylabel("Tiles")
    a2.set_title("Paired per-tile difference (deployed)", loc="left", pad=13)
    a2.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    a2.legend(loc="upper left", fontsize=6.5)
    style.panel_label(a1, "a"); style.panel_label(a2, "b")
    footnote(fig, FOOT + ".")
    style.save(fig, FIG_DIR, "fig_eval_per_tile")

# 11g. Collapse diagnosis on real validation data
if "collapsed" in ROLES:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(style.PAGE_W, 2.3), gridspec_kw={"width_ratios": [1.25, 1]})
    bins = np.linspace(0, 100, 51)
    gt_frac = PER_TILE[(PER_TILE.model == "final") & (PER_TILE.variant == "single_raw")].gt_frac * 100
    a1.hist(gt_frac, bins=bins, histtype="step", color=style.INK_2, lw=1.2, ls=":", label="ground truth")
    for role in ("collapsed", "final"):
        pf = PER_TILE[(PER_TILE.model == role) & (PER_TILE.variant == "single_raw")].positive_frac * 100
        a1.hist(pf, bins=bins, histtype="step", color=C[role], lw=1.5, label=L[role])
    a1.axvline(20, color=style.MUTED, lw=0.8, ls="--")
    a1.text(21, a1.get_ylim()[1] * 0.9, "collapse gate (20%)", fontsize=6.5, color=style.MUTED)
    a1.set_xlabel("Pixels predicted as road per tile (%)"); a1.set_ylabel("Tiles")
    a1.set_title("Predicted road fraction per tile (native)", loc="left")
    a1.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    a1.legend(loc="upper right", fontsize=6.5)
    roles2 = [r for r in ("collapsed", "run1", "final") if r in ROLES]
    yy = np.arange(len(roles2))[::-1] * 1.0
    for yi, role in zip(yy, roles2):
        s = 100 * TRAIN_PROTOCOL[role]["cldice_score"]
        i = 100 * TRAIN_PROTOCOL[role]["iou"]
        a2.text(1, yi + 0.36, L[role], fontsize=6.8, color=style.INK, fontweight="bold", va="bottom")
        a2.plot(s, yi + 0.12, "s", ms=6, color=C[role], mec="white", mew=1.0)
        a2.plot(i, yi - 0.14, "o", ms=6, mfc="white", mec=C[role], mew=1.5)
        a2.text(s + 2.5, yi + 0.12, f"soft clDice {s:.0f}", va="center", fontsize=6.4, color=style.INK_2)
        a2.text(i + 2.5, yi - 0.14, f"IoU {i:.1f}", va="center", fontsize=6.4, color=style.INK_2)
    a2.set_yticks([]); a2.grid(axis="y", visible=False)
    a2.set_ylim(-0.45, yy.max() + 0.75)
    a2.set_xlim(0, 118); a2.set_xlabel("Training-protocol score (%)")
    a2.set_title("What checkpoint selection saw", loc="left")
    style.panel_label(a1, "a"); style.panel_label(a2, "b")
    footnote(fig, FOOT + ". (b): soft clDice = 1 \u2212 logged clDice loss; 256\u00b2-resized tiles as in training.")
    style.save(fig, FIG_DIR, "fig_eval_collapse")

In [ ]:
# ── 12. Qualitative results: error maps on representative tiles ──
PROB_CMAP = LinearSegmentedColormap.from_list("pb", ["#ffffff", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])


def hex_rgb(h):
    return np.array([int(h[i:i + 2], 16) for i in (1, 3, 5)]) / 255.0


def error_map(P, G):
    img = np.ones((*G.shape, 3))
    img[P & G] = hex_rgb(style.TP_COLOR)
    img[P & ~G] = hex_rgb(style.FP_COLOR)
    img[~P & G] = hex_rgb(style.FN_COLOR)
    return img


@torch.no_grad()
def deployed_mask(model, rgb):
    x = (rgb.astype(np.float32) / 255.0 - np.array(IMNET_MEAN, np.float32)) / np.array(IMNET_STD, np.float32)
    _, pt = forward_probs(model, torch.from_numpy(x.transpose(2, 0, 1))[None].to(DEVICE), True)
    return binarize_all(pt[0, 0].cpu().numpy(), ["full"])["full"]


def _render_qualitative(panels, heads, name, subtitle, wide):
    """panels: [(tile_id, [(image, iou_or_None, is_mask), ...])]. wide=False puts tiles in rows
    (paper, full page); wide=True puts tiles in columns (slides)."""
    n_t, n_k = len(panels), len(heads)
    rows, cols = (n_k, n_t) if wide else (n_t, n_k)
    fig_w = style.PAGE_W
    fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_w / cols * rows + 0.45))
    axes = np.atleast_2d(axes)
    for t_i, (tid, items) in enumerate(panels):
        for k_i, (img, iou, is_mask) in enumerate(items):
            ax = axes[k_i, t_i] if wide else axes[t_i, k_i]
            if is_mask:
                ax.imshow(img, cmap=ListedColormap(["#ffffff", style.INK]), interpolation="nearest")
            else:
                ax.imshow(img, interpolation="nearest")
            if iou is not None:
                ax.text(0.03, 0.03, f"IoU {iou:.1f}", transform=ax.transAxes, fontsize=6.5, color=style.INK,
                        va="bottom", bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))
            style.image_axes(ax)
            for s in ax.spines.values():
                s.set_visible(True); s.set_color(style.GRID); s.set_linewidth(0.5)
    if wide:
        for t_i, (tid, _) in enumerate(panels):
            axes[0, t_i].set_title(tid, loc="left", fontsize=7, color=style.MUTED)
        for k_i, h in enumerate(heads):
            axes[k_i, 0].set_ylabel(h, fontsize=7.2, color=style.INK, rotation=90, labelpad=4)
    else:
        for t_i, (tid, _) in enumerate(panels):
            axes[t_i, 0].set_ylabel(tid, fontsize=6.5, color=style.MUTED)
        for ax, h in zip(axes[0], heads):
            ax.set_title(h, loc="left", fontsize=7.2)
    fig.subplots_adjust(wspace=0.03, hspace=0.05, bottom=0.06)
    fig.legend(handles=[Patch(color=style.TP_COLOR, label="correct road"), Patch(color=style.FP_COLOR, label="false road"),
                        Patch(color=style.FN_COLOR, label="missed road")],
               loc="upper center", bbox_to_anchor=(0.5, 0.055), ncol=3, fontsize=6.8)
    fig.text(0.02, 0.012, subtitle, fontsize=6.5, color=style.MUTED, va="top")
    style.save(fig, FIG_DIR, name + ("_wide" if wide else ""))


def qualitative(tiles, name, subtitle):
    roles_q = [r for r in ("baseline", "final") if r in ROLES]
    models = {r: load_mobilevit_checkpoint(CHECKPOINTS[r]["path"], device=DEVICE)[0] for r in roles_q}
    iou_lookup = PER_TILE.set_index(["tile", "model", "variant"]).iou
    panels = []
    for tid in tiles:
        rgb, G = load_tile(tid)
        items = [(rgb, None, False), (G, None, True)]
        for role in roles_q:
            items.append((error_map(deployed_mask(models[role], rgb), G),
                          100 * iou_lookup[(tid, role, DEPLOYED)], False))
        panels.append((tid, items))
    heads = ["Input", "Ground truth"] + [L[r] + "\n(deployed)" for r in roles_q]
    for wide in (False, True):
        _render_qualitative(panels, heads, name, subtitle, wide)
    del models
    torch.cuda.empty_cache()


_q = PER_TILE[(PER_TILE.model == "final") & (PER_TILE.variant == DEPLOYED) & (PER_TILE.gt_pixels >= 1000)].dropna(subset=["iou"])
_q = _q.sort_values("iou").reset_index(drop=True)
QUANTILES = [0.1, 0.3, 0.5, 0.7, 0.9]
Q_TILES = []
for q in QUANTILES:   # nearest not-yet-used tile to each percentile (never shows a tile twice)
    order = sorted(range(len(_q)), key=lambda i: abs(i - q * (len(_q) - 1)))
    pick = next((i for i in order if _q.tile[i] not in Q_TILES), None)
    if pick is not None:
        Q_TILES.append(_q.tile[pick])
qualitative(Q_TILES, "fig_eval_qualitative",
            "Tiles at the 10/30/50/70/90th percentile of the proposed model's per-tile IoU (not hand-picked).")
_hi = [t for t, b in TILE_TERTILE.items() if b == 2]
CANOPY_TILES = sorted(random.Random(7).sample(_hi, min(3, len(_hi))))
qualitative(CANOPY_TILES, "fig_eval_qualitative_canopy",
            f"Three tiles drawn at random (seed 7) from the highest canopy-cover tertile.")

In [ ]:
# ── 13. Training curves from attached training logs (if any) ──
for k, log in enumerate([l for l in TRAINING_LOGS if l["has_iou"]]):
    ep = log["epochs"]
    e = np.array([r["epoch"] for r in ep])
    iou = np.array([r["val/iou"] for r in ep]) * 100
    fig, axes = plt.subplots(1, 3, figsize=(style.PAGE_W, 2.2))
    a1, a2, a3 = axes
    a1.plot(e, [r["train/epoch_loss"] for r in ep], color=style.BLUE, label="train")
    a1.plot(e, [r["val/epoch_loss"] for r in ep], color=style.ORANGE, label="validation")
    a1.set_title("Composite loss", loc="left"); a1.set_xlabel("Epoch"); a1.legend()
    a2.plot(e, iou, color=style.BLUE, label="IoU")
    a2.plot(e, np.array([r["val/precision"] for r in ep]) * 100, color=style.AQUA, label="precision")
    b = int(np.argmax(iou))
    a2.plot(e[b], iou[b], "o", ms=5, color=style.BLUE, mec="white", mew=1.0)
    late = e[b] > (e[0] + e[-1]) / 2
    a2.annotate(f"best {iou[b]:.1f} (ep. {e[b]})", (e[b], iou[b]), xytext=(-5 if late else 5, 6),
                textcoords="offset points", ha="right" if late else "left", fontsize=6.5, color=style.INK_2)
    a2.set_title("Validation (256\u00b2 protocol)", loc="left"); a2.set_xlabel("Epoch"); a2.set_ylabel("%"); a2.legend(loc="center right")
    pf = np.array([r["val/positive_frac"] for r in ep]) * 100
    a3.plot(e, pf, color=style.BLUE)
    a3.axhline(20, color=style.MUTED, lw=0.8, ls="--"); a3.text(e[-1], 20.5, "collapse gate", ha="right", fontsize=6.5, color=style.MUTED)
    a3.set_title("Predicted road fraction", loc="left"); a3.set_xlabel("Epoch"); a3.set_ylabel("%")
    a3.set_ylim(0, max(25, pf.max() * 1.15))
    for ax, letter in zip(axes, "abc"):
        style.panel_label(ax, letter)
    fig.subplots_adjust(wspace=0.38)
    footnote(fig, f"Logged during training: {os.path.basename(os.path.dirname(log['path']))}/"
                  f"{os.path.basename(log['path'])}. Validation used 256\u00b2-resized tiles.")
    style.save(fig, FIG_DIR, f"fig_training_curves_{k + 1}")

In [ ]:
# ── 14. Package everything for download ──
zip_out = os.path.join(WORK, "paper_results.zip")
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        for f in files:
            full = os.path.join(root, f)
            zf.write(full, os.path.relpath(full, WORK))
print(f"{zip_out}: {os.path.getsize(zip_out) / 2**20:.1f} MB | split verified: {SPLIT_VERIFIED}")
print(sorted(os.listdir(FIG_DIR)))
if IN_KAGGLE:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(zip_out, WORK)))